In [ ]:
# Imports
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from collections import Counter
import json

# Cargar datos comprimidos
df = pd.read_csv("df_cluster_simple_birch_k4.csv.gz", compression="gzip", low_memory=False)

# Filtrar solo enfermedades clasificadas
df = df[df['cluster_k4'] != 'No clasificado'].copy()

# Normalizar cluster_k4: convertir a string sin decimales
df['cluster_k4'] = df['cluster_k4'].astype(float).astype(int).astype(str)

# Cargar diccionarios desde JSON
with open("cluster_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

grupo_dict = metadata["grupo_dict"]
descripcion_dict = metadata["descripcion_dict"]

# Aplicar mapeos
df['grupo'] = df['cluster_k4'].map(grupo_dict)
df['descripción'] = df['cluster_k4'].map(descripcion_dict)

# Preparar opciones para el widget
df_unique = df[['ORPHAcode', 'Preferred_term_unificado']].drop_duplicates()
df_unique['display'] = df_unique['Preferred_term_unificado'] + ' (ORPHA-' + df_unique['ORPHAcode'].astype(str) + ')'
selection_dict = dict(zip(df_unique['display'], df_unique['ORPHAcode']))

# Dropdown widget
dropdown_enfermedad = widgets.Combobox(
    placeholder='Selecciona una enfermedad...',
    options=sorted(selection_dict.keys()),
    description='Enfermedad:',
    ensure_option=True,
    layout=widgets.Layout(width='70%')
)

# Output
output = widgets.Output()

# Lógica al seleccionar enfermedad
def on_select(change):
    output.clear_output()
    seleccion = change['new']
    
    if seleccion in selection_dict:
        orpha = selection_dict[seleccion]
        subset = df[df['ORPHAcode'] == orpha]
        info = subset.iloc[0]
        nombre = info['Preferred_term_unificado']
        cluster = info['cluster_k4']
        grupo = info['grupo']
        descripcion = info['descripción']

        # Calcular top 10 fenotipos del clúster
        cluster_df = df[df['cluster_k4'] == cluster]
        hpo_counts = cluster_df['HPO_Term'].value_counts().head(10)
        top_hpos_df = hpo_counts.reset_index()
        top_hpos_df.columns = ['Fenotipo', 'Frecuencia en el clúster']

        with output:
            print(f"📌 Enfermedad: {nombre}")
            print(f"🧬 ORPHAcode: {orpha}")
            print(f"🔹 Clúster asignado: {cluster}")
            print(f"🔸 Grupo: {grupo}")
            print(f"📝 Descripción del grupo:\n{descripcion}\n")
            print("🔍 10 fenotipos más frecuentes en el clúster:")
            display(top_hpos_df)

# Enlazar dropdown con función
dropdown_enfermedad.observe(on_select, names='value')

# Mostrar interfaz
display(dropdown_enfermedad, output)

C:\Users\maria\AppData\Local\Temp\ipykernel_37660\3077988947.py:8: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("df_cluster_simple_birch_k4.csv.gz", compression="gzip")


Combobox(value='', description='Enfermedad:', ensure_option=True, layout=Layout(width='70%'), options=('10q22.…

Output()